In [1]:
!uv sync
!source ../venv/bin/activate


Resolved 70 packages in 5ms
Uninstalled 26 packages in 292ms
 - appnope==0.1.4
 - asttokens==3.0.1
 - comm==0.2.3
 - debugpy==1.8.19
 - decorator==5.2.1
 - executing==2.2.1
 - ipykernel==7.1.0
 - ipython==9.9.0
 - ipython-pygments-lexers==1.1.1
 - jedi==0.19.2
 - jupyter-client==8.8.0
 - jupyter-core==5.9.1
 - matplotlib-inline==0.2.1
 - nest-asyncio==1.6.0
 - parso==0.8.5
 - pexpect==4.9.0
 - platformdirs==4.5.1
 - prompt-toolkit==3.0.52
 - ptyprocess==0.7.0
 - pure-eval==0.2.3
 - pygments==2.19.2
 - pyzmq==27.1.0
 - stack-data==0.6.3
 - tornado==6.5.4
 - traitlets==5.14.3
 - wcwidth==0.2.14


In [6]:
import sys
from pathlib import Path

parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))


In [7]:
from scripts.run_extract_circuit_features import extract_circuits_features
from pandas import DataFrame

features = extract_circuits_features()
print(features)
df_init = DataFrame(features)

df = df_init.T
df = df.reset_index()

cols = df.columns.tolist()
cols[0] = "circuit"
df.columns = cols

print(df)

df = df.rename(columns={
            'circuit': 'Circuit',
            'num_qubits': 'Qubits',
            'two_qubit_gate_count': 'Two-qubit gates',
            'circuit_depth': 'Depth',
            'graph_diameter': 'Diameter',
            'average_degree': 'Average degree',
            'graph_density': 'Edge density',
            'interaction_variance': 'Interaction variance',
            'max_interactions_per_edge': 'Max interactions per edge',
        })

df['Qubits'] = df['Qubits'].astype(int)
df['Diameter'] = df['Diameter'].astype(int)
df['Depth'] = df['Depth'].astype(int)
df['Diameter'] = df['Diameter'].astype(int)
df['Two-qubit gates'] = df['Two-qubit gates'].astype(int)
df['Max interactions per edge'] = df['Max interactions per edge'].astype(int)

df_sorted = df.sort_values(by='Qubits', ascending=True)
print(df_sorted)


Available algorithm: demo_random
Available algorithm: liH
Available algorithm: h2
Available algorithm: qaoa
Available algorithm: qaoa_ansatz
Available algorithm: efficient_su2
{'demo_random': {'num_qubits': 9, 'two_qubit_gate_count': 60, 'circuit_depth': 73, 'graph_diameter': 4, 'average_degree': 2.888888888888889, 'graph_density': 0.3611111111111111, 'interaction_variance': 6.390532544378699, 'max_interactions_per_edge': 8}, 'liH': {'num_qubits': 12, 'two_qubit_gate_count': 6023, 'circuit_depth': 7328, 'graph_diameter': 2, 'average_degree': 9.666666666666666, 'graph_density': 0.8787878787878788, 'interaction_variance': 23859.820749108207, 'max_interactions_per_edge': 474}, 'h2': {'num_qubits': 4, 'two_qubit_gate_count': 30, 'circuit_depth': 42, 'graph_diameter': 1, 'average_degree': 3.0, 'graph_density': 1.0, 'interaction_variance': 16.0, 'max_interactions_per_edge': 9}, 'qaoa': {'num_qubits': 5, 'two_qubit_gate_count': 6, 'circuit_depth': 5, 'graph_diameter': 2, 'average_degree': 2.4

In [8]:
from pandas import DataFrame
from typing import List
import numpy as np

def escape_latex(text):
    return str(text).replace('_', r'\_')

def format_floats(num: float) -> str:
    """Format time with comma as decimal separator."""
    return f"{num:.3f}".replace('.', ',')

def prepare_value_for_latex(value) -> str:
    """Prepare different types of values for LaTeX output."""
    if isinstance(value, (float, np.floating)):
        return format_floats(value)
    return escape_latex(value)

# currently only supports basic tables
def df_to_table(df: DataFrame, table_name: str, table_label: str) -> str:
    latex_output: List[str] = []

    latex_output.append(r"\begin{table}[H]")
    latex_output.append(r"\centering")
    latex_output.append(r"\caption{" + table_name + "}")
    latex_output.append(r"\label{" + table_label + "}")
    latex_output.append(r"\begin{tabular}{|" + "|".join(["c"] * len(df.columns)) + r"|}")
    latex_output.append(r"\hline")
    latex_output.append(" & ".join([r"\textbf{"+x+"}" for x in list(df.columns.values)]) + r" \\")
    latex_output.append(r"\hline")
    for _, row in df.iterrows():
        latex_output.append(" & ".join([str(prepare_value_for_latex(item)) for item in row.values]) + r" \\ \hline")
    latex_output.append(r"\end{tabular}")
    latex_output.append(r"\end{table}")

    return '\n'.join(latex_output)


In [9]:
latex_table = df_to_table(df_sorted, table_name="Features of the tested quantum circuits", table_label="tab:circuit_features")
print(latex_table)

\begin{table}[H]
\centering
\caption{Features of the tested quantum circuits}
\label{tab:circuit_features}
\begin{tabular}{|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Circuit} & \textbf{Qubits} & \textbf{Two-qubit gates} & \textbf{Depth} & \textbf{Diameter} & \textbf{Average degree} & \textbf{Edge density} & \textbf{Interaction variance} & \textbf{Max interactions per edge} \\
\hline
h2 & 4 & 30 & 42 & 1 & 3,000 & 1,000 & 16,000 & 9 \\ \hline
qaoa & 5 & 6 & 5 & 2 & 2,400 & 0,600 & 0,000 & 1 \\ \hline
qaoa\_ansatz & 5 & 6 & 7 & 2 & 2,400 & 0,600 & 0,000 & 1 \\ \hline
efficient\_su2 & 5 & 4 & 8 & 4 & 1,600 & 0,400 & 0,000 & 1 \\ \hline
demo\_random & 9 & 60 & 73 & 4 & 2,889 & 0,361 & 6,391 & 8 \\ \hline
liH & 12 & 6023 & 7328 & 2 & 9,667 & 0,879 & 23859,821 & 474 \\ \hline
\end{tabular}
\end{table}
